<a href="https://colab.research.google.com/github/thanosleggis/Thesis/blob/main/notebooks/BERT%2CSHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- ΒΗΜΑ 1: Setup ---
!pip install transformers

import torch
import torch.nn as nn
import numpy as np
import scipy.io
from torch.utils.data import Dataset, DataLoader
from transformers import BertConfig, BertModel
from sklearn.model_selection import train_test_split
from google.colab import drive

# Συνδέουμε το Google Drive (θα σου ζητήσει έγκριση)
drive.mount('/content/drive')

In [ ]:
import zipfile
import os
import scipy.io
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# --- ΒΗΜΑ 2: Αποσυμπίεση & Φόρτωση Πραγματικών Δεδομένων ---

# 1. Ρυθμίσεις Αρχείων
zip_path = '/content/drive/MyDrive/DREAMER.zip'   # Το αρχείο στο Drive σου
extract_path = '/content/dreamer_data'            # Φάκελος για εξαγωγή

# 2. Αποσυμπίεση (Unzip)
if not os.path.exists(extract_path):
    print(f"⏳ Αποσυμπίεση του {zip_path}... (Περίμενε λίγο)")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(" Η αποσυμπίεση ολοκληρώθηκε!")
    except FileNotFoundError:
        print(" ΣΦΑΛΜΑ: Το αρχείο DREAMER.zip δεν βρέθηκε. Ελέγξτε το path στο Google Drive.")
        # Σταματάμε εδώ αν δεν βρεθεί αρχείο
        raise
else:
    print(" Τα αρχεία είναι ήδη αποσυμπιεσμένα.")

# 3. Εντοπισμός του .mat
mat_path = os.path.join(extract_path, 'DREAMER.mat')

# 4. Συνάρτηση που "ξεψαχνίζει" το δύσκολο αρχείο DREAMER
def parse_dreamer_real_data(mat_path, seq_len=256):
    print(f" Ανάγνωση δεδομένων από: {mat_path}")
    mat = scipy.io.loadmat(mat_path)

    # Μπαίνουμε βαθιά στη δομή (Structs)
    dreamer = mat['DREAMER'][0, 0]
    data = dreamer['Data']

    all_eeg = []
    all_labels = []

    # Το DREAMER έχει 23 εθελοντές
    num_subjects = 23

    for sub in range(num_subjects):
        sub_data = data[0, sub]

        # EEG από τα 18 βίντεο (trials)
        # Διαδρομή: Data -> Subject -> EEG -> stimuli
        eeg_stimuli = sub_data['EEG'][0, 0]['stimuli'][0, 0]

        # Scores (Valence)
        # Διαδρομή: Data -> Subject -> ScoreValence
        valence_scores = sub_data['ScoreValence'][0, 0]

        for trial in range(18):
            # Παίρνουμε το σήμα (Time x 14 κανάλια)
            raw_signal = eeg_stimuli[trial, 0]

            # Κόβουμε τα τελευταία 256 σημεία (πιο δυνατό συναίσθημα)
            if raw_signal.shape[0] >= seq_len:
                signal_slice = raw_signal[-seq_len:, :]
            else:
                # Padding αν είναι μικρό
                pad = np.zeros((seq_len - raw_signal.shape[0], 14))
                signal_slice = np.vstack((pad, raw_signal))

            all_eeg.append(signal_slice)

            # Label: 0 (Low Valence) ή 1 (High Valence)
            score = valence_scores[trial, 0]
            label = 1 if score >= 3.0 else 0
            all_labels.append(label)

    # Μετατροπή σε Tensors
    X = np.array(all_eeg, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)

    print(f"🎉 Επιτυχία! Φορτώθηκαν {X.shape[0]} δείγματα από {num_subjects} άτομα.")
    return X, y

# 5. Δημιουργία Dataset
class DreamerDataset(Dataset):
    def __init__(self, mat_path):
        self.data, self.labels = parse_dreamer_real_data(mat_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx]), torch.tensor(self.labels[idx])

# 6. Δημιουργία DataLoader
# Τρέχουμε τη διαδικασία μόνο αν βρέθηκε το αρχείο
if os.path.exists(mat_path):
    dataset = DreamerDataset(mat_path)
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True)
    print("🚀 Ο Loader είναι έτοιμος για εκπαίδευση!")
else:
    print(" Δεν βρέθηκε το DREAMER.mat μετά την αποσυμπίεση.")

In [ ]:
# --- ΒΗΜΑ 3: Ορισμός Μοντέλου ---

class DreamerEEGModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        # 1. Προσαρμογή των 14 καναλιών στο μέγεθος του BERT
        self.eeg_projection = nn.Linear(14, config.hidden_size)

        # 2. Ο κορμός (Backbone) του BERT
        self.transformer = BertModel(config)

        # 3. Ταξινομητής (Classifier)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # x shape: [Batch, Time, 14] -> [16, 128, 14]

        # Προβολή (Projection)
        x_projected = self.eeg_projection(x)

        # Πέρασμα από BERT (χωρίς embeddings κειμένου, βάζουμε τα δικά μας)
        outputs = self.transformer(inputs_embeds=x_projected)

        # Mean Pooling: Παίρνουμε τον μέσο όρο όλων των χρονικών στιγμών
        # outputs.last_hidden_state shape: [Batch, Time, Hidden]
        mean_pooling = torch.mean(outputs.last_hidden_state, dim=1)

        # Dropout και Classification
        x_out = self.dropout(mean_pooling)
        logits = self.classifier(x_out)

        return logits

# Ρυθμίσεις (Config)
HF_MODEL_LINK = "google-bert/bert-base-uncased"
config = BertConfig.from_pretrained(HF_MODEL_LINK)

# Προσαρμογές για το Dataset μας
config.num_labels = 2           # 2 Κλάσεις (High/Low)
config.hidden_size = 256        # Μικρό μέγεθος για ταχύτητα
config.num_attention_heads = 4
config.num_hidden_layers = 4    # Λίγα layers για να μην αργεί

# Δημιουργία αντικειμένου
model = DreamerEEGModel(config)

# Μεταφορά σε GPU (αν υπάρχει)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Το μοντέλο είναι έτοιμο στην: {device}")

In [ ]:
# --- ΒΗΜΑ 4: Εκπαίδευση ---

# Optimizer (AdamW είναι ο στάνταρ για BERT)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Loss Function (CrossEntropy για Classification)
criterion = nn.CrossEntropyLoss()

epochs = 5  # Πόσες φορές θα δει τα δεδομένα

print("Έναρξη εκπαίδευσης...")

for epoch in range(epochs):
    model.train() # Βάζουμε το μοντέλο σε mode εκπαίδευσης
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        # Μεταφορά δεδομένων στην GPU/CPU
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Μηδενισμός gradients
        optimizer.zero_grad()

        # 2. Forward pass (Πρόβλεψη)
        outputs = model(inputs)

        # 3. Υπολογισμός σφάλματος
        loss = criterion(outputs, labels)

        # 4. Backward pass (Ενημέρωση βαρών)
        loss.backward()
        optimizer.step()

        # Στατιστικά
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] -> Loss: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%")

print("Η εκπαίδευση ολοκληρώθηκε!")

In [ ]:
# --- ΒΗΜΑ 6: SHAP Analysis (Διόρθωση Επιλογής Κλάσης) ---
print("\n SHAP Analysis (GradientExplainer)...")

# ... (ο υπόλοιπος κώδικας για τον explainer παραμένει ίδιος) ...

# 4. Επεξεργασία του Shape (Εδώ έγινε το σφάλμα)
# Αν το shap_plot έχει σχήμα (2, 14, 256):
# index 0 -> Low Valence
# index 1 -> High Valence

if shap_plot.ndim == 3:
    print(f" Βρέθηκαν {shap_plot.shape[0]} κλάσεις. Επιλέγω την κλάση 1 (High Valence).")
    shap_to_plot = shap_plot[1] # Παίρνουμε μόνο τη δεύτερη κλάση -> (14, 256)
else:
    shap_to_plot = shap_plot

print(f" Τελικό Shape για imshow: {shap_to_plot.shape}")

# 5. Οπτικοποίηση
plt.figure(figsize=(15, 7))
v_max = np.max(np.abs(shap_to_plot))

# Τώρα το shap_to_plot είναι (14, 256) και θα δουλέψει!
plt.imshow(shap_to_plot, aspect='auto', cmap='RdBu_r', vmin=-v_max, vmax=v_max)

plt.colorbar(label='SHAP Value (Impact on High Valence)')
plt.title("SHAP Feature Importance (BERT EEG Model)")
plt.ylabel("EEG Channels (14)")
plt.xlabel("Time Samples (256)")
plt.show()

In [ ]:
!pip install lime
from lime import lime_tabular
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- ΒΗΜΑ 11: LIME Analysis (Safe Version) ---
print("\n Running LIME (Local Explanations)...")

# 0. Έλεγχος αν το μοντέλο και τα δεδομένα υπάρχουν
if 'model' not in locals():
    print("ΣΦΑΛΜΑ: Το 'model' δεν βρέθηκε. Τρέξε πρώτα το κελί του ορισμού του μοντέλου!")
else:
    model.eval()

    # 1. Συνάρτηση πρόβλεψης (Προσαρμοσμένη για το BERT σχήμα [Batch, Time, Channels])
    def predict_proba(x_np):
        # Το LIME δίνει flat δεδομένα (N, 3584). Τα γυρνάμε σε (N, 256, 14)
        x_tensor = torch.tensor(x_np, dtype=torch.float32).reshape(-1, 256, 14).to(device)
        with torch.no_grad():
            logits = model(x_tensor)
            probs = torch.softmax(logits, dim=1)
            return probs.cpu().numpy()

    # 2. Προετοιμασία δεδομένων
    # Παίρνουμε ένα δείγμα για εξήγηση από τον loader
    try:
        batch = next(iter(train_loader))
        inputs, labels = batch
        test_sample_lime = inputs[0:1] # Το πρώτο δείγμα του batch
    except NameError:
        print(" ΣΦΑΛΜΑ: Το 'train_loader' δεν βρέθηκε.")
        raise

    # Flattening για το LIME (από 256x14 σε 3584)
    X_train_flat = inputs.view(inputs.shape[0], -1).cpu().numpy()
    X_test_flat = test_sample_lime.view(1, -1).cpu().numpy()

    # 3. Ορισμός του Tabular Explainer
    # Το EEG αντιμετωπίζεται ως πίνακας χαρακτηριστικών (Time x Channels)
    explainer_lime = lime_tabular.LimeTabularExplainer(
        training_data=X_train_flat,
        mode="classification",
        class_names=['Low Valence', 'High Valence'],
        feature_names=[f"T{s}_Ch{c}" for s in range(256) for c in range(14)],
        verbose=True
    )

    # 4. Εξήγηση συγκεκριμένου δείγματος
    print("⏳ Υπολογισμός εξήγησης (μπορεί να πάρει 1-2 λεπτά)...")
    exp = explainer_lime.explain_instance(
        X_test_flat[0],
        predict_proba,
        num_features=15 # Τα 15 πιο καθοριστικά χαρακτηριστικά
    )

    # 5. Οπτικοποίηση
    print("\n Αποτελέσματα LIME:")
    exp.as_pyplot_figure()
    plt.title("LIME: Σημαντικότητα Χρονικών Σημείων & Καναλιών")
    plt.xlabel("Επίδραση στην Πρόβλεψη (Weight)")
    plt.show()